# Étape 3 - Croissance à unité constante

Notebook de travail pour l'étape 3 des consignes :

> Mesurer la croissance à unité constante : apparier les baux consécutifs d'une même unité sur `sPropCode + sUnitCode`,
> filtrer les écarts non comparables et annualiser.

**Principe :** au lieu de comparer le loyer médian d'une année à l'autre (qui mesure surtout le *mix*, voir section 2e du
notebook final), on compare **chaque appartement à lui-même** : son loyer actuel contre son loyer au bail précédent.
Même appartement, même immeuble, même taille - ce qui reste, c'est la variation de prix.

Trois étapes :
1. **Apparier** : pour chaque bail, retrouver le bail précédent de la même unité.
2. **Filtrer** : écarter les paires qui ne sont pas comparables.
3. **Annualiser** : ramener chaque variation à son équivalent sur un an.

Une fois validé, le contenu sera reporté dans la section 3 de `equinoxe-2026.ipynb`.

## 0. Configuration et chargement

Mêmes constantes et même renommage que le notebook final (section 2i), pour que le code se transfère tel quel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)

LEASES_PATH = "equinoxe_lease_history.csv"

UNIT_KEY = ["sPropCode", "sUnitCode"]       # clé sûre : 1 pour 1 avec hUnit (section 2a)
WRONG_UNIT_KEY = ["sSite", "sUnitCode"]     # clé fautive, utilisée uniquement pour montrer l'erreur

RENEWAL, TURNOVER = 1, 0
FIRST_ANALYSIS_YEAR = 2019
LAST_OBSERVED_YEAR = 2025
DAYS_PER_YEAR = 365.25
MONTHS_PER_YEAR = 12
PCT = 100

# Filtres d'appariement (justifiés en section 2 de ce notebook)
MIN_GAP_YEARS = 0.5     # exclut les paires issues d'un bail de 6 mois (voir 2b)
MAX_GAP_YEARS = 2.5     # exclut les comparaisons trop anciennes

# Tranches d'écart (années) pour décrire la distribution des écarts
GAP_BINS_YEARS = [0, 0.5, 0.95, 1.05, 1.6, 2.5, np.inf]

LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sConcession": "has_concession",
    "sRenewal": "is_renewal",
    "sTermSeq": "unit_lease_rank",
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",
    "sLeaseTerm": "lease_term_label_unreliable",
}

leases = pd.read_csv(LEASES_PATH)
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    leases[col] = pd.to_datetime(leases[col])
leases = leases.rename(columns=LEASE_COLUMN_RENAMES)
leases["lease_start_year"] = leases["lease_start"].dt.year
print(f"{len(leases):,} baux, {leases.groupby(UNIT_KEY).ngroups:,} unités")

## 1. Apparier les baux consécutifs d'une même unité

On trie les baux par unité puis par date de début. Pour chaque bail, le « bail précédent » est la ligne juste au-dessus
**dans la même unité** (`groupby(UNIT_KEY).shift(1)`). Le premier bail connu d'une unité n'a pas de précédent : il ne forme
pas de paire.

On conserve, pour le bail précédent, les deux loyers (contractuel et effectif), la date de début et la durée.

In [ ]:
PREVIOUS_LEASE_COLUMNS = ["contract_rent", "effective_rent", "lease_start", "lease_term_months"]


def pair_consecutive_leases(lease_df, unit_key):
    """Associe chaque bail au bail précédent de la même unité (colonnes préfixées par `prev_`).

    Retourne uniquement les baux qui ont un bail précédent.
    """
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired


all_pairs = pair_consecutive_leases(leases, UNIT_KEY)
print(f"{len(leases):,} baux - {leases.groupby(UNIT_KEY).ngroups:,} premiers baux (sans précédent) = {len(all_pairs):,} paires")

Exemple : l'historique complet d'une unité et les paires qu'il produit.

In [ ]:
example_unit = all_pairs.groupby(UNIT_KEY).size().idxmax()
example_history = all_pairs.set_index(UNIT_KEY).loc[[example_unit]]
example_history[["prev_lease_start", "prev_contract_rent", "lease_start", "contract_rent",
                 "gap_years", "is_renewal"]].round(3)

### Ce qui se passe avec la mauvaise clé (`sSite + sUnitCode`)

Les consignes interdisent d'apparier sur `sSite + sUnitCode`. On le fait quand même, une seule fois, pour **montrer l'effet** :
deux appartements différents qui partagent un site et un numéro sont traités comme un seul, et leurs baux s'entremêlent.

In [ ]:
wrong_pairs = pair_consecutive_leases(leases, WRONG_UNIT_KEY)
key_comparison = pd.DataFrame({
    "sPropCode + sUnitCode": ((all_pairs["contract_rent"] / all_pairs["prev_contract_rent"]) - 1) * PCT,
    "sSite + sUnitCode (faux)": ((wrong_pairs["contract_rent"] / wrong_pairs["prev_contract_rent"]) - 1) * PCT,
}).describe().round(2)
key_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
raw_growth_right = (all_pairs["contract_rent"] / all_pairs["prev_contract_rent"] - 1) * PCT
raw_growth_wrong = (wrong_pairs["contract_rent"] / wrong_pairs["prev_contract_rent"] - 1) * PCT
bins = np.linspace(raw_growth_wrong.quantile(0.01), raw_growth_wrong.quantile(0.99), 80)
ax.hist(raw_growth_wrong, bins=bins, alpha=0.6, label="sSite + sUnitCode (faux)")
ax.hist(raw_growth_right, bins=bins, alpha=0.6, label="sPropCode + sUnitCode")
ax.set_xlabel("variation du loyer contractuel entre deux baux consécutifs (%)")
ax.set_ylabel("nombre de paires")
ax.set_title("La mauvaise clé fabrique de fausses hausses et de fausses baisses")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture :** la médiane change peu (la majorité des unités n'est pas en collision), mais la mauvaise clé produit des
variations absurdes - des « baisses » d'environ 60 % et des « hausses » de plus de 150 % - parce qu'elle compare un appartement
à un autre appartement du même site qui porte le même numéro. Elle crée aussi des paires qui n'existent pas
(3 384 au lieu de 3 241). L'écart-type est environ 7 fois plus grand et la moyenne est tirée vers le haut (5,8 % au lieu de 4,6 %).
Avec la bonne clé, toutes les variations restent dans une plage plausible (environ -8 % à +24 %).

**Conséquence :** on apparie exclusivement sur `UNIT_KEY = ["sPropCode", "sUnitCode"]`.

## 2. Filtrer les écarts non comparables

Une paire n'est utile que si les deux loyers sont **comparables**. On examine d'abord la distribution réelle des écarts
entre deux baux consécutifs avant de fixer les seuils.

### 2a. Distribution des écarts entre deux baux

In [ ]:
all_pairs["gap_bucket"] = pd.cut(all_pairs["gap_years"], bins=GAP_BINS_YEARS)
all_pairs["raw_growth_pct"] = (all_pairs["contract_rent"] / all_pairs["prev_contract_rent"] - 1) * PCT

gap_distribution = all_pairs.groupby("gap_bucket", observed=True).agg(
    pairs=("gap_years", "size"),
    median_prev_term_months=("prev_lease_term_months", "median"),
    median_raw_growth_pct=("raw_growth_pct", "median"))
gap_distribution["share_pct"] = gap_distribution["pairs"] / len(all_pairs) * PCT
print(all_pairs["gap_years"].describe().round(3), end="\n\n")
gap_distribution.round(2)

In [ ]:
gap_minus_previous_term_months = all_pairs["gap_years"] * MONTHS_PER_YEAR - all_pairs["prev_lease_term_months"]
print("Écart réel - durée du bail précédent (en mois) :")
print(gap_minus_previous_term_months.describe().round(2))

**Lecture :**

- **81 % des paires sont espacées d'un an** (0,95 à 1,05 an) : le cas standard d'un bail de 12 mois suivi du suivant.
- L'écart entre deux baux correspond presque toujours à la **durée du bail précédent** (différence médiane ≈ 0 mois) :
  les baux s'enchaînent sans trou. Un écart de 6 mois signifie donc un bail précédent de 6 mois, un écart de 18 mois
  un bail de 18 mois.
- Aucun écart ne dépasse environ 2,1 ans, et il n'y a **aucun doublon** (section 2a du notebook final).
- **Observation clé :** la hausse *brute* est d'environ 4,3 à 4,9 % **quel que soit l'écart**. Que le bail précédent ait duré
  6, 12 ou 18 mois, le loyer augmente à peu près du même pourcentage au bail suivant. La hausse se fait **par bail**,
  pas au prorata du temps écoulé.

### 2b. Choix des filtres

| Filtre | Règle | Justification |
|---|---|---|
| Écart minimal | `gap_years > MIN_GAP_YEARS` (0,5 an) | Les paires de moins de 6 mois viennent toutes d'un bail de 6 mois (2 % des paires). Ce ne sont pas des doublons, mais leur hausse brute (environ 4,4 % comme les autres) annualisée sur 6 mois deviendrait environ 9 %/an : elle n'est pas comparable à un renouvellement annuel. On les exclut et on mesure l'effet de ce choix en section 5. |
| Écart maximal | `gap_years < MAX_GAP_YEARS` (2,5 ans) | Au-delà, l'unité a pu être rénovée ou rester vacante longtemps : la variation mélange plusieurs années. Dans ces données, le seuil n'élimine aucune paire (écart maximal environ 2,1 ans), mais il protège la méthode si on l'applique à d'autres données. |
| Loyers positifs | `prev_rent > 0` et `rent > 0` | Un loyer nul rendrait le ratio infini. Aucun cas ici (section 2 du notebook final), mais le filtre est gardé par sécurité. |

On applique les filtres un par un pour voir combien de paires chacun retire.

In [ ]:
def filter_comparable_pairs(paired, price_col, verbose=False):
    """Garde les paires dont l'écart est comparable et dont les deux loyers sont positifs."""
    prev_price_col = f"prev_{price_col}"
    steps = [
        ("écart > MIN_GAP_YEARS", paired["gap_years"] > MIN_GAP_YEARS),
        ("écart < MAX_GAP_YEARS", paired["gap_years"] < MAX_GAP_YEARS),
        ("loyers positifs", (paired[prev_price_col] > 0) & (paired[price_col] > 0)),
    ]
    keep = pd.Series(True, index=paired.index)
    funnel = [("paires appariées", len(paired))]
    for label, condition in steps:
        keep &= condition
        funnel.append((label, int(keep.sum())))
    if verbose:
        funnel_table = pd.DataFrame(funnel, columns=["étape", "paires restantes"])
        funnel_table["paires retirées"] = -funnel_table["paires restantes"].diff().fillna(0).astype(int)
        print(funnel_table.to_string(index=False))
    return paired[keep].copy()


comparable_pairs = filter_comparable_pairs(all_pairs, "contract_rent", verbose=True)

## 3. Annualiser

Deux paires peuvent afficher la même hausse sur des durées différentes : +6 % en 12 mois n'est pas +6 % en 18 mois.
Annualiser, c'est répondre à la question : *quelle hausse **par an** aurait produit ce résultat?*

$$\text{croissance annualisée} = \left(\frac{\text{loyer}}{\text{loyer précédent}}\right)^{1/\text{écart en années}} - 1$$

On utilise une puissance plutôt qu'une division, car les hausses se **composent** comme des intérêts :
+3,96 %/an pendant 1,5 an redonne bien +6 % au total ($1{,}0396^{1{,}5} = 1{,}06$).

Exemple : 1 000 $ → 1 060 $ en 18 mois : $(1060/1000)^{1/1{,}5} - 1 = 3{,}96$ %/an, et non 6 %.

In [ ]:
def annualized_growth_pct(paired, price_col):
    """Croissance annualisée (%) entre le bail précédent et le bail courant."""
    price_ratio = paired[price_col] / paired[f"prev_{price_col}"]
    return (price_ratio ** (1 / paired["gap_years"]) - 1) * PCT


comparable_pairs["growth_pct"] = annualized_growth_pct(comparable_pairs, "contract_rent")

annualization_effect = comparable_pairs.groupby("gap_bucket", observed=True).agg(
    pairs=("growth_pct", "size"),
    median_raw_growth_pct=("raw_growth_pct", "median"),
    median_annualized_growth_pct=("growth_pct", "median"))
annualization_effect.round(2)

**Lecture :** pour les paires à un an (81 %), annualiser ne change rien. Pour les paires de 18 à 24 mois, la hausse brute
d'environ 4,5 % devient environ 2,5 %/an : c'est la bonne lecture pour un **budget annuel**, puisque ce loyer est resté
inchangé pendant plus d'un an. L'effet global reste faible (voir la sensibilité en section 5).

## 4. Résultats

On regroupe les trois étapes dans une seule fonction, appliquée au loyer contractuel puis au loyer effectif.
Chaque paire est rattachée à l'année de début du **nouveau** bail (`lease_start_year`) : c'est l'année où le nouveau loyer
entre en vigueur. On résume chaque année par la **médiane**, robuste aux quelques paires extrêmes.

In [ ]:
def same_unit_growth(lease_df, price_col):
    """Croissance annualisée du loyer à unité constante : apparier, filtrer, annualiser."""
    paired = pair_consecutive_leases(lease_df, UNIT_KEY)
    comparable = filter_comparable_pairs(paired, price_col)
    comparable["growth_pct"] = annualized_growth_pct(comparable, price_col)
    return comparable


def median_growth_table(pairs, by):
    return (pairs.groupby(by)["growth_pct"]
            .agg(pairs="size", median="median")
            .round(2))


contract_pairs = same_unit_growth(leases, "contract_rent")
effective_pairs = same_unit_growth(leases, "effective_rent")
analysis_years = list(range(FIRST_ANALYSIS_YEAR, LAST_OBSERVED_YEAR + 1))

growth_by_year = pd.concat({
    "contractuel": median_growth_table(contract_pairs, "lease_start_year"),
    "effectif": median_growth_table(effective_pairs, "lease_start_year"),
}, axis=1).loc[analysis_years]
growth_by_year

In [ ]:
naive_median_growth = leases.groupby("lease_start_year")["contract_rent"].median().pct_change() * PCT

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(analysis_years, naive_median_growth.loc[analysis_years], "o:", lw=2, label="médiane naïve de tous les baux")
ax.plot(analysis_years, growth_by_year[("contractuel", "median")], "o-", lw=2, label="même unité - loyer contractuel")
ax.plot(analysis_years, growth_by_year[("effectif", "median")], "s-", lw=2, label="même unité - loyer effectif")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("année de début du nouveau bail")
ax.set_ylabel("croissance annuelle (%)")
ax.set_title("Croissance à unité constante vs médiane naïve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4b. Par type de bail, par province et par immeuble

Ces découpages préparent les étapes suivantes (renouvellements vs relocations, règles du Québec vs de l'Ontario,
prévision par immeuble). On affiche toujours le **nombre de paires** : une médiane sur peu de paires n'est pas fiable.

In [ ]:
BREAKDOWN_FIRST_YEAR = 2022   # découpages détaillés limités aux 4 dernières années, pour rester lisibles
recent_years = [year for year in analysis_years if year >= BREAKDOWN_FIRST_YEAR]

by_lease_type = (effective_pairs[effective_pairs["lease_start_year"].isin(recent_years)]
                 .assign(lease_type=lambda df: df["is_renewal"].map({RENEWAL: "renouvellement", TURNOVER: "relocation"}))
                 .pivot_table(index="lease_start_year", columns="lease_type", values="growth_pct",
                              aggfunc=["size", "median"]))
print("Loyer effectif - par type de bail :")
print(by_lease_type.round(2), end="\n\n")

by_province = (effective_pairs[effective_pairs["lease_start_year"].isin(recent_years)]
               .pivot_table(index="lease_start_year", columns="sState", values="growth_pct",
                            aggfunc=["size", "median"]))
print("Loyer effectif - par province :")
print(by_province.round(2))

In [ ]:
by_building = (effective_pairs[effective_pairs["lease_start_year"].isin(recent_years)]
               .pivot_table(index="sBuilding", columns="lease_start_year", values="growth_pct",
                            aggfunc=["size", "median"]))
by_building.round(2)

## 5. Sensibilité aux choix de méthode

Chaque choix (seuil minimal, annualisation, médiane) pourrait changer le résultat. On recalcule la croissance effective
2023-2025 en modifiant **un choix à la fois**. Si les chiffres bougent peu, la méthode est robuste.

In [ ]:
SENSITIVITY_YEARS = [2023, 2024, 2025]
INCLUDE_SIX_MONTH_GAP_YEARS = 0.4      # seuil alternatif qui inclut les baux de 6 mois
TRIM_SHARE = 0.05                      # moyenne tronquée : retire 5 % des paires de chaque côté


def trimmed_mean(values):
    low, high = values.quantile([TRIM_SHARE, 1 - TRIM_SHARE])
    return values[values.between(low, high)].mean()


paired_effective = pair_consecutive_leases(leases, UNIT_KEY)
paired_effective["raw_growth_pct"] = (paired_effective["effective_rent"] / paired_effective["prev_effective_rent"] - 1) * PCT
paired_effective["annualized_growth_pct"] = annualized_growth_pct(paired_effective, "effective_rent")
within_max_gap = paired_effective["gap_years"] < MAX_GAP_YEARS
reference = paired_effective[within_max_gap & (paired_effective["gap_years"] > MIN_GAP_YEARS)]
with_six_month = paired_effective[within_max_gap & (paired_effective["gap_years"] > INCLUDE_SIX_MONTH_GAP_YEARS)]

variants = {
    "référence (médiane, annualisée, écart > 0,5 an)": reference.groupby("lease_start_year")["annualized_growth_pct"].median(),
    "inclure les baux de 6 mois": with_six_month.groupby("lease_start_year")["annualized_growth_pct"].median(),
    "sans annualisation": reference.groupby("lease_start_year")["raw_growth_pct"].median(),
    "moyenne au lieu de médiane": reference.groupby("lease_start_year")["annualized_growth_pct"].mean(),
    "moyenne tronquée à 5 %": reference.groupby("lease_start_year")["annualized_growth_pct"].apply(trimmed_mean),
}
sensitivity = pd.DataFrame(variants).loc[SENSITIVITY_YEARS].T.round(2)
sensitivity["écart max vs référence"] = (sensitivity.sub(sensitivity.iloc[0])).abs().max(axis=1).round(2)
sensitivity

**Lecture :** aucune variante ne s'écarte de plus de 0,25 point de la référence. Inclure les baux de 6 mois,
renoncer à l'annualisation ou remplacer la médiane par une moyenne ne change pas la conclusion : **la méthode est robuste**.
La moyenne simple est légèrement plus haute que la médiane, tirée par quelques fortes hausses - ce qui justifie la médiane.

## 6. Conclusions

**Méthode retenue**

| Étape | Choix | Effet |
|---|---|---|
| Apparier | bail précédent de la même unité, clé `sPropCode + sUnitCode` | 3 241 paires sur 1 061 unités |
| Filtrer | écart entre 0,5 et 2,5 ans, loyers positifs | 62 paires retirées (baux de 6 mois), 3 179 conservées |
| Annualiser | (loyer / loyer précédent) ^ (1 / écart en années) - 1 | ne change rien pour 81 % des paires (écart d'un an) |
| Résumer | médiane par année de début du nouveau bail | robuste : toutes les variantes à moins de 0,25 point |

**Ce que disent les résultats**

1. **La vraie hausse est bien plus modeste que la médiane naïve.** En 2023, la médiane naïve affiche +10,8 % alors qu'un même
   appartement n'a augmenté que de 2,3 % (contractuel) ou 2,0 % (effectif).
2. **Contractuel et effectif divergent depuis 2024.** En 2025 : +7,0 % au bail, +3,6 % réellement perçu - l'écart vient des
   concessions (section 2f du notebook final).
3. **2023 est une année de ralentissement dans tous les immeubles** (environ 2 % partout). C'est un effet d'année,
   pas d'immeuble : à rapprocher des taux du TAL et de l'IPC à l'étape des données externes.
4. **Renouvellements et relocations s'inversent.** En 2022-2023, les renouvellements augmentent plus vite; en 2024-2025,
   ce sont les relocations (2025 : 5,7 % contre 2,9 % en effectif). À analyser à l'étape 5.
5. **Le Québec et l'Ontario évoluent de façon très proche** en 2024-2025 (écart de moins de 0,3 point).
6. **Certaines médianes par immeuble ne sont pas fiables** : Le Carlyle 2023 (1 paire), Westpark 2024 (7 paires).
   Une prévision par immeuble devra s'appuyer sur la tendance du portefeuille pour ces cas.

**À reporter dans le notebook final (section 3) :** les fonctions `pair_consecutive_leases`, `filter_comparable_pairs`,
`annualized_growth_pct` et `same_unit_growth`, la démonstration de la mauvaise clé, la distribution des écarts,
le tableau des résultats et la sensibilité.